# Injury Intelligence - Classification

Author: Angela Lekivetz

This notebook performs classification of injury narratives based on OSHA report sourced from [OSHA Accident and Injury Data](https://www.kaggle.com/datasets/ruqaiyaship/osha-accident-and-injury-data-1517). The narratives have been cleaned and lemmatized using [spaCy](https://spacy.io/) in the `notebooks/1_eda.ipynb` notebook.

Two models will be trained: 
- A baseline model is trained using TF-IDF + Logistic Regression
- DistilBERT fine-tune

The baseline model provides a benchmark for comparison, where if DistilBERT doesn't meaningfully improve the performance, the added complexity isn't justified. All experiments will be tracked with MLflow for reproducibility and comparison.

In [28]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_scheduler
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the Dataset

The cleaned and lemmatized dataset from `1_eda.ipynb` is loaded, containing columns such as abstract text, event type and keywords, and the cleaned and lemmatized text.

 `lemma_text` is used as input for the TF-IDF baseline model, while `clean_text` is used for the DistilBERT model.

In [29]:
df = pd.read_csv('../data/osha_filtered.csv')
df.head()

,Abstract Text,Event type,Event Keywords,Nature of Injury,Part of Body,text_len,clean_text,lemma_text
0,"At 9:00 a.m. on August 10, 2017, an employee w...",Caught in or between,"FINGER,MECHANICAL POWER PRESS,AMPUTATION,GUARD","Amputation, Crushing",Fingers,73,an employee was operating ton bliss coin knuck...,employee operate ton bliss coin knuckle mechan...
1,"At 9:45 a.m. on July 17, 2017, an employee was...",Caught in or between,"CAUGHT IN,DRIVE SHAFT,RESIDENTIAL CONSTRUCTION...",Dislocation,Fingers,107,an employee was using battery operated drill t...,employee battery operate drill drill hole wood...
2,"At 7:30 a.m. on June 30, 2017, an employee was...",Other,"AMPUTATED,EXPLOSION,FIREWORKS",Fire Burn,Hand,29,an employee was inserting match fuze into fire...,employee insert match fuze firework charge fir...
3,"At 2:00 p.m. on June 30, 2017, an employee was...",Fall (from elevation),"RIB,ROOF,HEAD,FALL PROTECTION,FALL,COLLARBONE,...",Serious Fall/Strike,Head,121,an employee was installing self adhering membr...,employee instal self adhere membrane roof slop...
4,"At 12:20 p.m. on June 23, 2017, an employee wa...",Struck-by,"STRUCK BY,TRUCK,BRAIN,NECK,FRACTURE,UNSTABLE LOAD","Bruising, Contusion",Neck,53,an employee was delivering load of plywood to ...,employee deliver load plywood project site emp...


## 2. Prepare Data

This step prepares the data for training by applying a label encoder to the classifier column `Event type`.

The data is then split 80/20 into training and testing sets, using a constant seed for reproducibility.

In [30]:
# Encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['Event type'])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['lemma_text'],
    df['label'],
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

In [31]:
print(X_train.shape, X_test.shape)
print(pd.Series(y_train).value_counts())
print(le.classes_)

(3541,) (886,)
label
2    940
5    904
1    897
3    505
4    153
0    142
Name: count, dtype: int64
['Card-vascular/resp. fail.' 'Caught in or between'
 'Fall (from elevation)' 'Other' 'Shock' 'Struck-by']


## 3. Baseline Model (TF-IDF + Logistic Regression)

Our baseline model is a Term Frequency-Inverse Document Frequency (TF-IDF) + Logistic Regression model. TF-IDF converts text into numbers by measuring the relative importance of each word in the text, penalizing words that are common across the entire set. Logistic regression is then used to predict the probability of an a class occurring based on the TF-IDF features.

This baseline model is simple, fast, and interpretable. 

In [35]:
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('injury-classification')

with mlflow.start_run(run_name='tfidf-logreg'):
    # Vectorize
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)

    # Train
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=SEED)
    clf.fit(X_train_vec, y_train)

    # Evaluate
    y_pred = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)

    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Log
    mlflow.log_param('model', 'tfidf-logreg')
    mlflow.log_param('max_features', 20000)
    mlflow.log_param('ngram_range', '(1,2)')
    mlflow.log_metric('accuracy', acc)
    mlflow.sklearn.log_model(clf, 'model')

2026/05/20 20:33:31 INFO mlflow.tracking.fluent: Experiment with name 'injury-classification' does not exist. Creating a new experiment.
2026/05/20 20:33:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 20:33:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.7822
                           precision    recall  f1-score   support

Card-vascular/resp. fail.       0.65      0.42      0.51        36
     Caught in or between       0.80      0.75      0.78       224
    Fall (from elevation)       0.91      0.92      0.92       235
                    Other       0.67      0.65      0.66       126
                    Shock       0.97      0.84      0.90        38
                Struck-by       0.69      0.79      0.73       227

                 accuracy                           0.78       886
                macro avg       0.78      0.73      0.75       886
             weighted avg       0.78      0.78      0.78       886



## 4. DistilBERT Fine-tuned Model

We then use DistilBERT, a smaller version of BERT (Bidirectional Encoder Representations from Transformers), to fine-tune the model on our dataset. BERT understands contextual language, and therefore we can adapt its general language understanding capabilities to our specific dataset.

In [41]:
class InjuryDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = InjuryDataset(X_train.tolist(), y_train.to_list(), tokenizer, MAX_LEN)
test_dataset = InjuryDataset(X_test.tolist(), y_test.tolist(), tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
model.to(device)

with mlflow.start_run(run_name='distilbert-finetune'):
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=LR)
    num_training_steps = EPOCHS * len(train_loader)
    scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    mlflow.log_params({'model': MODEL_NAME, 'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN})

    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        mlflow.log_metric('train_loss', avg_loss, step=epoch)
        print(f'Epoch {epoch+1} — Loss: {avg_loss:.4f}')

    # Evaluation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

    bert_acc = accuracy_score(all_labels, all_preds)
    print(f'\nDistilBERT Accuracy: {bert_acc:.4f}')
    print(classification_report(all_labels, all_preds, target_names=le.classes_))
    mlflow.log_metric('accuracy', bert_acc)

    # Save model
    model.save_pretrained('../data/distilbert_injury')
    tokenizer.save_pretrained('../data/distilbert_injury')
    print('Model saved.')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15941.26it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 15482.85it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert

Epoch 1 — Loss: 1.0567


Epoch 2: 100%|██████████| 222/222 [03:15<00:00,  1.13it/s]


Epoch 2 — Loss: 0.5864


Epoch 3: 100%|██████████| 222/222 [04:27<00:00,  1.21s/it]


Epoch 3 — Loss: 0.4700


Evaluating: 100%|██████████| 56/56 [00:17<00:00,  3.26it/s]



DistilBERT Accuracy: 0.7935
                           precision    recall  f1-score   support

Card-vascular/resp. fail.       0.61      0.53      0.57        36
     Caught in or between       0.81      0.82      0.81       224
    Fall (from elevation)       0.91      0.93      0.92       235
                    Other       0.66      0.58      0.62       126
                    Shock       0.93      1.00      0.96        38
                Struck-by       0.72      0.75      0.74       227

                 accuracy                           0.79       886
                macro avg       0.77      0.77      0.77       886
             weighted avg       0.79      0.79      0.79       886



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.94it/s]

Model saved.


# 5. Compare Models

We see a modest improvement in accuracy when using the DistilBERT model (`79.35%`) vs the TF-IDF model (`78.22%`). This is expected on short, structured text. This gap isn't as pronounced as TF-IDF works well on short, consistent text with distinctive vocabulary such as "roof", "ladder", or "voltage". 

In [42]:
print(f'TF-IDF + LogReg accuracy : {acc:.4f}')
print(f'DistilBERT accuracy      : {bert_acc:.4f}')
print(f'Improvement              : +{(bert_acc - acc)*100:.2f}pp')

TF-IDF + LogReg accuracy : 0.7822
DistilBERT accuracy      : 0.7935
Improvement              : +1.13pp


## 6. Inference on Sample Narratives

To validate the model qualitatively, we test it on a few hand-crafted narratives with known expected classes. 

The model performs well on classes with distinctive vocabulary such as "caught in" and "fell from elevation", but struggles when narratives contain language associated with multiple event types.

In [44]:
sample_texts = [
    ("the employee was operating a mechanical press when their hand was caught in the point of operation", "Caught in or between"),
    ("an employee was delivering materials when a load of lumber fell from the truck and struck them in the head", "Struck-by"),
    ("an employee was working on a roof when they fell approximately 15 feet to the ground below", "Fall (from elevation)")
]

model.eval()
inputs = tokenizer([t[0] for t in sample_texts], truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)

for (text, expected), pred in zip(sample_texts, preds):
    print(f'Text:      {text[:80]}')
    print(f'Expected:  {expected}')
    print(f'Predicted: {le.classes_[pred]}')
    print()

Text:      the employee was operating a mechanical press when their hand was caught in the 
Expected:  Caught in or between
Predicted: Caught in or between

Text:      an employee was delivering materials when a load of lumber fell from the truck a
Expected:  Struck-by
Predicted: Other

Text:      an employee was working on a roof when they fell approximately 15 feet to the gr
Expected:  Fall (from elevation)
Predicted: Fall (from elevation)

